# Phase 3+4 integration — 5 additional seeds for n=10 (Colab)

Extends the report-029 5-seed run with 5 *new* seeds {3, 5, 7, 13, 19} for a total n=10 verification. Same config as report 029:
- `--success-threshold 0.3`
- `--no-reencode-discovered` (default after report 030)
- Death mechanism defaults (threshold=0.05, window=10 — never fired in any prior run, but passed explicitly for reproducibility)

**Why:** Report 031 diagnostic showed seed 23 is not architecturally pathological — same codebook geometry, same baseline rigidity, same Phase 4 candidate yield as seeds 1, 11, 2. Its negative ΔR@10 is a dynamics-trajectory outlier, not a substrate issue. With n=10 instead of n=5, the per-seed t-CI for ΔR@10 tightens by ~40%, likely excluding zero outright and closing blocker #1.

**Output paths** use the same `phase34_integrated_hebbian_st03_seed{N}` prefix as the original report-029 runs. Aggregation iterates over all 10 seeds.

**Prereqs:** codebook at `MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt`, A100 runtime, Run all (~10 min).

In [ ]:
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt'
dst_dir = 'reports/phase3c_reconstruction'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, f'{dst_dir}/phase3c_codebook_reconstruction.pt')
!ls -lh {dst_dir}/phase3c_codebook_reconstruction.pt

In [ ]:
# `datasets` for HF wikitext download. `torchhd` is no longer required
# (the codebase uses its own FHRR substrate; the prior install was dead
# weight and started failing on Blackwell runtimes due to wheel coverage).
!pip install -q datasets
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# Run 5 additional seeds {3, 5, 7, 13, 19} with report-029 config.
import subprocess, os, time
from pathlib import Path

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
log_root = Path('reports/phase34_integrated_hebbian_st03_ext5_colab')
log_root.mkdir(parents=True, exist_ok=True)

for seed in [3, 5, 7, 13, 19]:
    out_dir = f'reports/phase34_integrated_hebbian_st03_seed{seed}'
    log_path = log_root / f'seed{seed}.log'
    print(f'\n=== seed {seed} -> {out_dir} ===  ({time.strftime("%H:%M:%S")})')
    with open(log_path, 'w') as logf:
        proc = subprocess.run(
            ['python', 'experiments/19_phase34_integrated.py',
             '--updater-kind', 'hebbian',
             '--success-threshold', '0.3',
             '--no-reencode-discovered',
             '--death-threshold', '0.05',
             '--death-window', '10',
             '--seed', str(seed),
             '--device', 'cuda',
             '--output-dir', out_dir],
            stdout=logf, stderr=subprocess.STDOUT,
        )
    print(f'    exit={proc.returncode}  log={log_path}')
    !tail -8 {log_path}

In [ ]:
# Copy results back to Drive.
import shutil, os
dst = '/content/drive/MyDrive/neuro-ai/results'
os.makedirs(dst, exist_ok=True)
for seed in [3, 5, 7, 13, 19]:
    src = f'reports/phase34_integrated_hebbian_st03_seed{seed}'
    if os.path.isdir(src):
        shutil.copytree(src, f'{dst}/phase34_integrated_hebbian_st03_seed{seed}',
                        dirs_exist_ok=True)
shutil.copytree('reports/phase34_integrated_hebbian_st03_ext5_colab',
                f'{dst}/phase34_integrated_hebbian_st03_ext5_colab', dirs_exist_ok=True)
print('results in', dst)
!ls {dst}

In [ ]:
# Summary table for the 5 new seeds.
import json
from pathlib import Path

print(f'{"seed":>4} {"cond":>16} {"top1":>8} {"R@10":>8} {"cap_t05":>8} {"cands":>6} {"cons":>5} {"deaths":>6} {"drift":>9}')
for seed in [3, 5, 7, 13, 19]:
    p = Path(f'reports/phase34_integrated_hebbian_st03_seed{seed}/phase34_results.json')
    if not p.exists(): continue
    d = json.loads(p.read_text())
    for cond in ('baseline_static', 'phase3_reencode', 'phase3_phase4'):
        final = d['results'][cond][-1]
        cands = final.get('candidates_total', 0)
        cons = final.get('consolidations', 0)
        deaths = final.get('deaths_total', 0)
        drift = final.get('codebook_drift_from_initial', 0.0)
        print(f'{seed:>4} {cond:>16} {final["top1"]:>8.3f} {final["topk"]:>8.3f} '
              f'{final["cap_t_05"]:>8.3f} {cands:>6d} {cons:>5d} {deaths:>6d} {drift:>9.2e}')